In [0]:
%sql
DESCRIBE TABLE proyecto_final.bronze.tracking_envios;

In [0]:
%sql
SELECT *
FROM proyecto_final.bronze.tracking_envios
ORDER BY fecha_modificacion, evento_id
LIMIT 30;

In [0]:
%sql
SELECT
    COUNT(*) AS total_bronze,
    COUNT(DISTINCT evento_id) AS eventos_distintos,
    COUNT(DISTINCT tracking_id) AS envios_distintos,
    COUNT(DISTINCT venta_id) AS ventas_distintas,
    MIN(fecha_modificacion) AS primera_modificacion,
    MAX(fecha_modificacion) AS ultima_modificacion,
    SUM(CASE WHEN evento_id IS NULL THEN 1 ELSE 0 END)
        AS eventos_sin_id,
    SUM(CASE WHEN tracking_id IS NULL OR TRIM(tracking_id) = ''
        THEN 1 ELSE 0 END) AS tracking_id_vacios,
    SUM(CASE WHEN fecha_modificacion IS NULL THEN 1 ELSE 0 END)
        AS fechas_modificacion_nulas
FROM proyecto_final.bronze.tracking_envios;

In [0]:
%sql
SELECT
    'ESTADO_ENTREGA' AS tipo,
    COALESCE(estado_entrega, '<NULL>') AS valor_original,
    COUNT(*) AS total
FROM proyecto_final.bronze.tracking_envios
GROUP BY estado_entrega

UNION ALL

SELECT
    'COURIER' AS tipo,
    COALESCE(courier, '<NULL>') AS valor_original,
    COUNT(*) AS total
FROM proyecto_final.bronze.tracking_envios
GROUP BY courier

ORDER BY tipo, total DESC, valor_original;

In [0]:
%sql
WITH normalizados AS (
    SELECT
        evento_id,
        tracking_id,
        venta_id,
        fecha_modificacion,

        CASE
            WHEN LOWER(TRIM(estado_entrega)) = 'pendiente'
                THEN 'pendiente'
            WHEN LOWER(TRIM(estado_entrega)) = 'entregado'
                THEN 'entregado'
            WHEN LOWER(TRIM(estado_entrega)) = 'devuelto'
                THEN 'devuelto'
            WHEN LOWER(TRIM(estado_entrega)) IN
                 ('en transito', 'en_transito', 'en_camino', 'en camino')
                THEN 'en_transito'
            ELSE NULL
        END AS estado_normalizado

    FROM proyecto_final.bronze.tracking_envios
),
referencias AS (
    SELECT
        n.*,
        v.venta_id AS venta_encontrada
    FROM normalizados n
    LEFT JOIN proyecto_final.silver.ventas v
        ON n.venta_id = v.venta_id
),
eventos_reprocesados AS (
    SELECT
        tracking_id,
        estado_normalizado,
        fecha_modificacion,
        COUNT(*) AS apariciones
    FROM normalizados
    GROUP BY
        tracking_id,
        estado_normalizado,
        fecha_modificacion
),
envios AS (
    SELECT
        tracking_id,
        COUNT(DISTINCT venta_id) AS ventas_asociadas
    FROM normalizados
    GROUP BY tracking_id
)
SELECT
    (SELECT COUNT(*) FROM normalizados)
        AS total_bronze,

    (SELECT COUNT(*) FROM referencias
     WHERE venta_encontrada IS NULL)
        AS eventos_venta_huerfana,

    (SELECT COUNT(*) FROM normalizados
     WHERE estado_normalizado IS NULL)
        AS estados_invalidos,

    (SELECT COALESCE(SUM(apariciones - 1), 0)
     FROM eventos_reprocesados
     WHERE apariciones > 1)
        AS eventos_reprocesados,

    (SELECT COUNT(*) FROM envios
     WHERE ventas_asociadas > 1)
        AS tracking_con_multiples_ventas;

In [0]:
%sql
SELECT
    tracking_id,
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT venta_id) AS ventas_distintas,
    SORT_ARRAY(COLLECT_SET(venta_id)) AS ventas_asociadas,
    SORT_ARRAY(COLLECT_SET(estado_entrega)) AS estados_originales,
    MIN(fecha_modificacion) AS primer_evento,
    MAX(fecha_modificacion) AS ultimo_evento
FROM proyecto_final.bronze.tracking_envios
GROUP BY tracking_id
HAVING COUNT(DISTINCT venta_id) > 1
ORDER BY ventas_distintas DESC, tracking_id
LIMIT 30;

In [0]:
%sql
WITH eventos AS (
    SELECT
        t.evento_id,
        t.tracking_id,
        t.venta_id,
        t.estado_entrega,
        t.fecha_modificacion,
        CASE
            WHEN t.venta_id IS NULL
              OR TRIM(t.venta_id) = ''
            THEN 'VENTA_VACIA'
            WHEN v.venta_id IS NULL
            THEN 'VENTA_HUERFANA'
            ELSE 'VENTA_VALIDA'
        END AS estado_referencia
    FROM proyecto_final.bronze.tracking_envios t
    LEFT JOIN proyecto_final.silver.ventas v
        ON t.venta_id = v.venta_id
),
resumen_tracking AS (
    SELECT
        tracking_id,
        COUNT(DISTINCT CASE
            WHEN estado_referencia = 'VENTA_VALIDA'
            THEN venta_id
        END) AS ventas_validas,
        COUNT(DISTINCT CASE
            WHEN estado_referencia <> 'VENTA_VALIDA'
            THEN venta_id
        END) AS ventas_huerfanas,
        SUM(CASE
            WHEN estado_referencia <> 'VENTA_VALIDA'
            THEN 1 ELSE 0
        END) AS eventos_huerfanos
    FROM eventos
    GROUP BY tracking_id
)
SELECT
    (SELECT COUNT(*) FROM eventos
     WHERE estado_referencia = 'VENTA_VACIA')
        AS ventas_vacias,

    (SELECT COUNT(*) FROM eventos
     WHERE estado_referencia = 'VENTA_HUERFANA')
        AS eventos_venta_huerfana,

    (SELECT COUNT(*) FROM eventos
     WHERE estado_referencia = 'VENTA_VALIDA')
        AS eventos_venta_valida,

    (SELECT COUNT(*) FROM resumen_tracking
     WHERE ventas_validas > 0
       AND ventas_huerfanas > 0)
        AS tracking_mixtos,

    (SELECT COUNT(*) FROM resumen_tracking
     WHERE ventas_validas = 0
       AND eventos_huerfanos > 0)
        AS tracking_solo_huerfanos,

    (SELECT COUNT(*) FROM resumen_tracking
     WHERE ventas_validas > 1)
        AS tracking_multiples_ventas_validas;

In [0]:
%sql
WITH eventos_validos AS (
    SELECT
        t.evento_id,
        t.tracking_id,
        t.venta_id,
        t.fecha_modificacion,

        CASE
            WHEN LOWER(TRIM(t.estado_entrega)) = 'pendiente'
                THEN 'pendiente'
            WHEN LOWER(TRIM(t.estado_entrega)) = 'entregado'
                THEN 'entregado'
            WHEN LOWER(TRIM(t.estado_entrega)) = 'devuelto'
                THEN 'devuelto'
            WHEN LOWER(TRIM(t.estado_entrega)) IN (
                'en transito',
                'en_transito',
                'en_camino',
                'en camino'
            )
                THEN 'en_transito'
            ELSE NULL
        END AS estado_normalizado

    FROM proyecto_final.bronze.tracking_envios t
    INNER JOIN proyecto_final.silver.ventas v
        ON t.venta_id = v.venta_id
),
resumen_envios AS (
    SELECT
        tracking_id,
        MIN(CASE
            WHEN estado_normalizado = 'pendiente'
            THEN fecha_modificacion
        END) AS primer_pendiente,

        MIN(CASE
            WHEN estado_normalizado = 'entregado'
            THEN fecha_modificacion
        END) AS primer_entregado,

        SUM(CASE
            WHEN estado_normalizado = 'devuelto'
            THEN 1 ELSE 0
        END) AS eventos_devolucion

    FROM eventos_validos
    GROUP BY tracking_id
)
SELECT
    COUNT(*) AS envios_con_eventos_validos,

    SUM(CASE
        WHEN primer_pendiente IS NOT NULL
        THEN 1 ELSE 0
    END) AS envios_con_pendiente,

    SUM(CASE
        WHEN primer_entregado IS NOT NULL
        THEN 1 ELSE 0
    END) AS envios_con_entregado,

    SUM(CASE
        WHEN primer_pendiente IS NOT NULL
         AND primer_entregado IS NOT NULL
         AND primer_entregado >= primer_pendiente
        THEN 1 ELSE 0
    END) AS entregas_con_tiempo_calculable,

    SUM(CASE
        WHEN primer_entregado IS NOT NULL
         AND primer_pendiente IS NULL
        THEN 1 ELSE 0
    END) AS entregas_sin_pendiente,

    SUM(CASE
        WHEN primer_entregado < primer_pendiente
        THEN 1 ELSE 0
    END) AS entregas_antes_pendiente,

    SUM(CASE
        WHEN eventos_devolucion > 0
        THEN 1 ELSE 0
    END) AS envios_con_devolucion

FROM resumen_envios;

In [0]:
%sql
WITH eventos AS (
    SELECT
        t.tracking_id,
        t.venta_id,
        t.courier,
        t.fecha_modificacion,
        CASE
            WHEN LOWER(TRIM(t.estado_entrega)) = 'pendiente'
                THEN 'pendiente'
            WHEN LOWER(TRIM(t.estado_entrega)) = 'entregado'
                THEN 'entregado'
            WHEN LOWER(TRIM(t.estado_entrega)) = 'devuelto'
                THEN 'devuelto'
            WHEN LOWER(TRIM(t.estado_entrega)) IN (
                'en transito', 'en_transito',
                'en_camino', 'en camino'
            )
                THEN 'en_transito'
            ELSE NULL
        END AS estado_normalizado,
        CASE
            WHEN v.venta_id IS NOT NULL THEN 1
            ELSE 0
        END AS venta_valida
    FROM proyecto_final.bronze.tracking_envios t
    LEFT JOIN proyecto_final.silver.ventas v
        ON t.venta_id = v.venta_id
),
resumen AS (
    SELECT
        tracking_id,
        MAX(venta_valida) AS tiene_venta_valida,
        MAX(CASE WHEN venta_valida = 0 THEN 1 ELSE 0 END)
            AS tiene_venta_huerfana,
        COUNT(DISTINCT CASE WHEN venta_valida = 1
            THEN venta_id END) AS ventas_validas_distintas,
        COUNT(DISTINCT CASE WHEN venta_valida = 1
            THEN LOWER(TRIM(courier)) END) AS couriers_distintos,
        MIN(CASE
            WHEN venta_valida = 1
             AND estado_normalizado = 'pendiente'
            THEN fecha_modificacion
        END) AS primer_pendiente,
        MIN(CASE
            WHEN venta_valida = 1
             AND estado_normalizado = 'entregado'
            THEN fecha_modificacion
        END) AS primer_entregado
    FROM eventos
    GROUP BY tracking_id
)
SELECT
    COUNT(*) AS envios_con_venta_valida,

    SUM(CASE
        WHEN tiene_venta_huerfana = 1
        THEN 1 ELSE 0 END) AS envios_mixtos,

    SUM(CASE
        WHEN primer_entregado IS NOT NULL
         AND primer_pendiente IS NULL
        THEN 1 ELSE 0 END) AS entregas_sin_pendiente,

    SUM(CASE
        WHEN primer_entregado IS NOT NULL
         AND primer_pendiente IS NULL
         AND tiene_venta_huerfana = 1
        THEN 1 ELSE 0 END) AS sin_pendiente_en_envios_mixtos,

    SUM(CASE
        WHEN primer_entregado IS NOT NULL
         AND primer_pendiente IS NULL
         AND tiene_venta_huerfana = 0
        THEN 1 ELSE 0 END) AS sin_pendiente_no_mixtos,

    SUM(CASE
        WHEN couriers_distintos > 1
        THEN 1 ELSE 0 END) AS envios_con_varios_couriers

FROM resumen
WHERE tiene_venta_valida = 1;

In [0]:
%sql
WITH normalizados AS (
    SELECT
        tracking_id,
        CASE
            WHEN LOWER(TRIM(courier)) = 'servientrega'
                THEN 'Servientrega'
            WHEN LOWER(TRIM(courier)) = 'shalom'
                THEN 'Shalom'
            WHEN LOWER(TRIM(courier)) IN ('olva', 'olva courier')
                THEN 'Olva'
            ELSE NULL
        END AS courier_normalizado
    FROM proyecto_final.bronze.tracking_envios
),
resumen AS (
    SELECT
        tracking_id,
        COUNT(DISTINCT courier_normalizado)
            AS couriers_distintos
    FROM normalizados
    GROUP BY tracking_id
)
SELECT
    COUNT(*) AS total_envios,
    SUM(CASE WHEN couriers_distintos = 1
        THEN 1 ELSE 0 END) AS envios_un_courier,
    SUM(CASE WHEN couriers_distintos > 1
        THEN 1 ELSE 0 END) AS envios_varios_couriers,
    SUM(CASE WHEN couriers_distintos = 0
        THEN 1 ELSE 0 END) AS envios_sin_courier
FROM resumen;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Leer Bronze Tracking y referencias de Silver Ventas
df_bronze_tracking = spark.table(
    "proyecto_final.bronze.tracking_envios"
)

df_ventas_tracking = (
    spark.table("proyecto_final.silver.ventas")
    .select(F.col("venta_id").alias("venta_ref"))
    .distinct()
)

# 2. Normalizar estados y couriers
estado = F.lower(F.trim(F.col("estado_entrega")))
courier = F.lower(F.trim(F.col("courier")))

df_tracking_normalizado = (
    df_bronze_tracking
    .withColumn(
        "estado_normalizado",
        F.when(estado == "pendiente", "pendiente")
        .when(estado == "entregado", "entregado")
        .when(estado == "devuelto", "devuelto")
        .when(
            estado.isin(
                "en transito",
                "en_transito",
                "en_camino",
                "en camino"
            ),
            "en_transito"
        )
    )
    .withColumn(
        "courier_normalizado",
        F.when(courier == "servientrega", "Servientrega")
        .when(courier == "shalom", "Shalom")
        .when(
            courier.isin("olva", "olva courier"),
            "Olva"
        )
    )
)

# 3. Identificar eventos reprocesados
ventana_eventos = (
    Window.partitionBy(
        "tracking_id",
        "estado_normalizado",
        "fecha_modificacion"
    )
    .orderBy(F.col("evento_id").asc_nulls_last())
)

df_tracking_deduplicado = (
    df_tracking_normalizado
    .withColumn(
        "rn",
        F.row_number().over(ventana_eventos)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 4. Validar referencias contra Silver Ventas
df_tracking_enriquecido = (
    df_tracking_deduplicado
    .join(
        df_ventas_tracking,
        F.col("venta_id") == F.col("venta_ref"),
        "left"
    )
)

# 5. Aplicar reglas de calidad
df_calidad_tracking = (
    df_tracking_enriquecido
    .withColumn(
        "motivo_rechazo",
        F.concat_ws(
            ", ",
            F.when(
                F.col("evento_id").isNull(),
                "EVENTO_ID_INVALIDO"
            ),
            F.when(
                F.col("tracking_id").isNull()
                | (F.trim("tracking_id") == ""),
                "TRACKING_ID_INVALIDO"
            ),
            F.when(
                F.col("fecha_modificacion").isNull(),
                "FECHA_MODIFICACION_INVALIDA"
            ),
            F.when(
                F.col("estado_normalizado").isNull(),
                "ESTADO_INVALIDO"
            ),
            F.when(
                F.col("venta_ref").isNull(),
                "VENTA_HUERFANA"
            )
        )
    )
    .withColumn(
        "rechazado",
        F.length("motivo_rechazo") > 0
    )
)

# 6. Preparar los ocho campos de Silver
df_silver_tracking = (
    df_calidad_tracking
    .filter(~F.col("rechazado"))
    .select(
        F.col("evento_id").cast("int").alias("evento_id"),
        F.col("tracking_id").cast("string").alias("tracking_id"),
        F.col("venta_id").cast("string").alias("venta_id"),
        F.col("courier_normalizado").alias("courier"),
        F.col("estado_normalizado").alias("estado_entrega"),
        F.col("sucursal_origen").cast("string").alias("sucursal_origen"),
        F.col("fecha_actualizacion").cast("date").alias("fecha_actualizacion"),
        F.col("fecha_modificacion").cast("timestamp").alias("fecha_modificacion")
    )
)

# 7. Preparar los rechazos
df_rechazos_tracking = (
    df_calidad_tracking
    .filter(F.col("rechazado"))
)

# 8. Verificar resultados
total_bronze = df_bronze_tracking.count()
total_deduplicado = df_tracking_deduplicado.count()
total_silver = df_silver_tracking.count()
total_rechazos = df_rechazos_tracking.count()

print("Bronze:", total_bronze)
print("Sin eventos reprocesados:", total_deduplicado)
print("Eventos reprocesados eliminados:",
      total_bronze - total_deduplicado)
print("Silver preparada:", total_silver)
print("Cuarentena preparada:", total_rechazos)

assert total_bronze == 15139
assert total_deduplicado == 15139
assert total_silver == 14663
assert total_rechazos == 476

display(
    df_rechazos_tracking
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
TABLA_SILVER_TRACKING = "proyecto_final.silver.tracking_envios"

# Validar antes de escribir
assert df_silver_tracking.count() == 14663
assert df_rechazos_tracking.count() == 476

# Guardar eventos válidos en Silver
(
    df_silver_tracking.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER_TRACKING)
)

# Verificar registros almacenados
df_verificacion_tracking = spark.table(TABLA_SILVER_TRACKING)

total_cargado = df_verificacion_tracking.count()

eventos_distintos = (
    df_verificacion_tracking
    .select("evento_id")
    .distinct()
    .count()
)

envios_distintos = (
    df_verificacion_tracking
    .select("tracking_id")
    .distinct()
    .count()
)

print("Registros Silver:", total_cargado)
print("Eventos distintos:", eventos_distintos)
print("Envíos distintos:", envios_distintos)

assert total_cargado == 14663
assert eventos_distintos == 14663

display(
    df_verificacion_tracking
    .orderBy("tracking_id", "fecha_modificacion", "evento_id")
    .limit(20)
)

In [0]:
%sql
SELECT
    evento_id,
    tracking_id,
    venta_id,
    courier,
    estado_entrega,
    sucursal_origen,
    fecha_actualizacion,
    fecha_modificacion
FROM proyecto_final.silver.tracking_envios
ORDER BY tracking_id, fecha_modificacion, evento_id
LIMIT 30;

In [0]:
from pyspark.sql import functions as F

# Columnas originales de Bronze
columnas_bronze_tracking = df_bronze_tracking.columns

# Preparar registros rechazados
df_cuarentena_tracking = (
    df_rechazos_tracking
    .select(
        F.lit("tracking_envios").alias("fuente"),

        F.col("evento_id")
         .cast("string")
         .alias("clave_registro"),

        F.col("motivo_rechazo"),

        F.current_timestamp().alias("fecha_rechazo"),

        F.to_json(
            F.struct(
                *[
                    F.col(c)
                    for c in columnas_bronze_tracking
                ]
            ),
            options={"ignoreNullFields": "false"}
        ).alias("datos_originales"),

        F.lit("Azure SQL Database - dbo.TrackingEnvios")
         .alias("archivo_origen")
    )
)

# Verificar antes de cargar
total = df_cuarentena_tracking.count()

distintos = (
    df_cuarentena_tracking
    .select("clave_registro")
    .distinct()
    .count()
)

json_nulos = (
    df_cuarentena_tracking
    .filter(F.col("datos_originales").isNull())
    .count()
)

print("Registros preparados:", total)
print("Eventos distintos:", distintos)
print("JSON originales nulos:", json_nulos)

assert total == 476
assert distintos == 476
assert json_nulos == 0

display(
    df_cuarentena_tracking
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
from pyspark.sql import functions as F

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# 1. Validar datos preparados
assert df_cuarentena_tracking.count() == 476

# 2. Registrar conteos antes de la carga
print("ANTES DE LA CARGA")

df_antes = spark.table(TABLA_CUARENTENA)

display(
    df_antes
    .groupBy("fuente")
    .count()
    .orderBy("fuente")
)

# 3. Limpiar únicamente cargas anteriores de Tracking
spark.sql(f"""
    DELETE FROM {TABLA_CUARENTENA}
    WHERE fuente = 'tracking_envios'
""")

# 4. Insertar los eventos rechazados
(
    df_cuarentena_tracking
    .select(
        "fuente",
        "clave_registro",
        "motivo_rechazo",
        "fecha_rechazo",
        "datos_originales",
        "archivo_origen"
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLA_CUARENTENA)
)

# 5. Validar la carga
print("DESPUÉS DE LA CARGA")

df_despues = spark.table(TABLA_CUARENTENA)

display(
    df_despues
    .groupBy("fuente")
    .count()
    .orderBy("fuente")
)

total_tracking = (
    df_despues
    .filter(F.col("fuente") == "tracking_envios")
    .count()
)

assert total_tracking == 476

print("Tracking en cuarentena:", total_tracking)

In [0]:
%sql
SELECT
    fuente,
    clave_registro,
    motivo_rechazo,
    fecha_rechazo,
    datos_originales,
    archivo_origen
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'tracking_envios'
ORDER BY TRY_CAST(clave_registro AS INT)
LIMIT 30;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM proyecto_final.bronze.tracking_envios)
        AS total_bronze,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.tracking_envios)
        AS total_silver,

    (SELECT COUNT(DISTINCT evento_id)
     FROM proyecto_final.silver.tracking_envios)
        AS eventos_distintos_silver,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.cuarentena
     WHERE fuente = 'tracking_envios')
        AS total_cuarentena,

    (SELECT COUNT(DISTINCT clave_registro)
     FROM proyecto_final.silver.cuarentena
     WHERE fuente = 'tracking_envios')
        AS eventos_distintos_cuarentena,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.cuarentena
     WHERE fuente = 'tracking_envios'
       AND datos_originales IS NULL)
        AS json_originales_nulos;